# 02. Multi-Level Grouping in Pandas

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week10/02.Multi-Level-Grouping/notebooks/01_02.Multi-Level-Grouping.ipynb)

## Overview
Real-world data is inherently multi-dimensional. When analyzing university enrolments, hospital admissions, or retail sales, we rarely group by a single attribute. Instead, we group across multiple categorical hierarchies—such as **Campus and Faculty**, or **State and Store Format**.

In this notebook, we cover:
1. **Grouping by Multiple Keys**: Passing lists of columns to `df.groupby(['A', 'B'])`.
2. **The MultiIndex Result**: Navigating hierarchical row indexes.
3. **Hierarchical Slicing**: Extracting specific subsets using `.loc[]`.
4. **Pivoting Grouped Results with `.unstack()`**: Turning hierarchical rows into comparison tables.
5. **Cohort Proportions**: Computing within-group percentages using `.transform()`.

## 1. Setup: Faculty Enrolments Across Campuses

We construct a dataset of university faculties operating across Sydney, Melbourne, and Brisbane.

In [ ]:
import pandas as pd
import numpy as np

df_faculty = pd.DataFrame({
    'campus': [
        'North Sydney', 'North Sydney', 'North Sydney',
        'Melbourne', 'Melbourne', 'Melbourne',
        'Brisbane', 'Brisbane', 'Brisbane'
    ],
    'faculty': [
        'Health Sciences', 'Education & Arts', 'Law & Business',
        'Health Sciences', 'Education & Arts', 'Theology & Philosophy',
        'Health Sciences', 'Law & Business', 'Theology & Philosophy'
    ],
    'year': [2026] * 9,
    'enrolments': [1250, 820, 950, 1400, 910, 430, 890, 670, 310],
    'funding_kaud': [320.0, 180.5, 210.0, 380.0, 205.0, 115.0, 240.0, 160.0, 95.0],
    'satisfaction_pct': [88.5, 91.0, 84.0, 89.2, 92.5, 94.0, 87.0, 83.5, 93.0]
})

print("Faculty Enrolment Dataset:")
display(df_faculty)

## 2. Multi-Level Grouping

Passing a list of columns `['campus', 'faculty']` to `.groupby()` groups by distinct combinations of both dimensions.
The resulting DataFrame has a **MultiIndex** on its rows.

In [ ]:
grouped_fac = df_faculty.groupby(['campus', 'faculty'])[['enrolments', 'funding_kaud']].sum()
print("Multi-Level Grouping Output:")
display(grouped_fac)
print(f"\nIndex Type: {type(grouped_fac.index)}")
print(f"Index Levels: {grouped_fac.index.names}")

## 3. Slicing Hierarchical Groupings

We can slice hierarchical DataFrames using `.loc[]`:
- By top-level key: `.loc['North Sydney']`
- By specific tuple: `.loc[('Melbourne', 'Health Sciences')]`

In [ ]:
print("Slicing by Outer Group ('North Sydney'):")
display(grouped_fac.loc['North Sydney'])

print("\nSlicing Specific Tuple ('Melbourne', 'Health Sciences'):")
display(grouped_fac.loc[('Melbourne', 'Health Sciences')])

## 4. Unstacking Grouped Levels into Columns

Use `.unstack()` to rotate an inner index level into column headers for side-by-side comparison across campuses.

In [ ]:
# Unstack faculty into columns
enrolment_matrix = df_faculty.groupby(['campus', 'faculty'])['enrolments'].sum().unstack(fill_value=0)
print("Enrolments Unstacked by Faculty:")
display(enrolment_matrix)

## 5. Within-Group Calculations using `transform()`

To compute what percentage of each campus's total enrolment belongs to each faculty, we use `.transform('sum')` to broadcast campus totals across rows.

In [ ]:
df_analysis = df_faculty.copy()
df_analysis['campus_total'] = df_analysis.groupby('campus')['enrolments'].transform('sum')
df_analysis['enrolment_share_pct'] = (df_analysis['enrolments'] / df_analysis['campus_total'] * 100).round(1)

display(df_analysis[['campus', 'faculty', 'enrolments', 'campus_total', 'enrolment_share_pct']])

## 6. Practical Exercises

### Exercise 1: State and Store Format Revenue
Using the Australian retail sales dataset below:
1. Group by `['state', 'store_format']` and calculate total `revenue_kaud`.
2. Display the resulting MultiIndex Series.

In [ ]:
retail_sales = pd.DataFrame({
    'state': ['NSW', 'NSW', 'NSW', 'VIC', 'VIC', 'VIC', 'QLD', 'QLD'],
    'store_format': ['Flagship', 'Express', 'Suburban', 'Flagship', 'Express', 'Suburban', 'Flagship', 'Express'],
    'revenue_kaud': [450, 180, 220, 410, 195, 205, 320, 140],
    'transactions': [12000, 8500, 6200, 11500, 9200, 5800, 8900, 6100]
})

# --- Student Code Here ---
# state_format_rev = ...

# --- Solution ---
state_format_rev = retail_sales.groupby(['state', 'store_format'])['revenue_kaud'].sum()
display(state_format_rev)

### Exercise 2: Unstack Formats into Side-by-Side Columns
Take `state_format_rev` from Exercise 1 and unstack `store_format` to create a 2D table of revenues with `fill_value=0`.

In [ ]:
# --- Student Code Here ---
# format_matrix = ...

# --- Solution ---
format_matrix = state_format_rev.unstack(fill_value=0)
display(format_matrix)

## 7. Key Takeaways

1. **Multi-Column Grouping**: Pass a list `df.groupby(['key1', 'key2'])` to group by composite hierarchies.
2. **MultiIndex Results**: Grouping by multiple keys generates a hierarchical Index.
3. **`.unstack()` Integration**: Converts hierarchical rows into clean 2D comparison tables.
4. **`.transform()` for Normalisation**: Broadcasts group-level metrics across original rows to compute cohort percentages.